# Convolutional neural networks
**Machine Learning · Week 9 (Deep learning: images, sequences and attention) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w9d2.html

A **convolutional neural network** (CNN) stacks convolution layers, ReLU and pooling, and *learns* its
filters with backpropagation. To see the filters work, we need images bigger than 8 x 8: here is a dataset
of $20 \times 20$ pictures of four shapes, drawn anywhere, at any size, with a little noise.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SHAPES = ["ring", "square", "triangle", "plus"]

def make_shapes(n, size=20, seed=0):
    """n noisy size x size images of the four shapes, at random places, sizes and stroke widths."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size] + 0.5
    X = np.zeros((n, size, size), np.float32)
    y = rng.integers(0, 4, n)
    for i in range(n):
        r = rng.uniform(3.5, 6.5)                      # size
        cx, cy = rng.uniform(r + 0.5, size - r - 0.5, 2)   # center
        w = rng.uniform(0.9, 1.6)                      # stroke width
        dx, dy = xx - cx, yy - cy
        if y[i] == 0:                                  # distance from every pixel to the shape's outline
            d = np.abs(np.hypot(dx, dy) - r)
        elif y[i] == 1:
            d = np.abs(np.maximum(np.abs(dx), np.abs(dy)) - 0.85 * r)
        elif y[i] == 2:
            k = np.sqrt(3)
            edges = np.minimum.reduce([np.abs(dy - r / 2), np.abs(k * dx - dy - r) / 2, np.abs(-k * dx - dy - r) / 2])
            near = (dy <= r / 2 + w) & (k * dx - dy <= r + 2 * w) & (-k * dx - dy <= r + 2 * w)
            d = np.where(near, edges, 99)
        else:
            d = np.where((np.abs(dx) <= r) & (np.abs(dy) <= r), np.minimum(np.abs(dx), np.abs(dy)), 99)
        ink = np.clip(w - d + 0.5, 0, 1) * rng.uniform(0.7, 1.0)
        X[i] = np.clip(ink + rng.normal(0, 0.08, ink.shape), 0, 1)
    return X, y

X_train, y_train = make_shapes(1500, seed=0)
X_test, y_test = make_shapes(1000, seed=1)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4), layout="constrained")
for ax, img, label in zip(axes.flat, X_train, y_train):
    ax.imshow(img, cmap="Purples", vmin=0, vmax=1)
    ax.set_title(SHAPES[label], fontsize=11)
    ax.axis("off")
plt.show()

## Two networks
A plain network (week 5) flattens the 400 pixels. The CNN keeps the picture's shape: 8 filters of 3 x 3, ReLU,
max pooling to 10 x 10; then 16 filters, ReLU, pooling to 5 x 5; then one linear layer for the 4 scores.
Follow one batch through the CNN and watch the shapes: `(images, channels, height, width)`.

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(400, 64), nn.ReLU(), nn.Linear(64, 4))
torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 8 feature maps, 10 x 10
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 16 feature maps, 5 x 5
    nn.Flatten(), nn.Linear(16 * 5 * 5, 4))

count = lambda model: sum(p.numel() for p in model.parameters())
print(f"plain network: {count(mlp):,} weights")
print(f"CNN:           {count(cnn):,} weights\n")
x = torch.tensor(X_train[:32]).unsqueeze(1)          # add the channel axis: (32, 1, 20, 20)
print(f"{'input':10s} {tuple(x.shape)}")
for layer in cnn:
    x = layer(x)
    print(f"{type(layer).__name__:10s} {tuple(x.shape)}")

## Train both the same way
Same data, same optimizer, same number of passes. The CNN has about 9 times fewer weights.

In [ ]:
torch.set_num_threads(1)
Xtr, ytr = torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train)
Xte, yte = torch.tensor(X_test).unsqueeze(1), torch.tensor(y_test)

def train(model, epochs=15):
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(0)
    for epoch in range(epochs):
        order = torch.randperm(len(Xtr), generator=g)
        for i in range(0, len(Xtr), 64):
            batch = order[i:i + 64]
            opt.zero_grad()
            loss_fn(model(Xtr[batch]), ytr[batch]).backward()
            opt.step()
    return model

for name, model in [("plain network", mlp), ("CNN", cnn)]:
    t0 = time.perf_counter()
    train(model)
    with torch.no_grad():
        acc = (model(Xte).argmax(dim=1) == yte).float().mean().item()
    print(f"{name:14s} test accuracy {acc:.3f}   ({count(model):,} weights, {time.perf_counter() - t0:.1f} s)")

## What did the CNN learn?
Top: the 8 filters of the first layer, learned from scratch. Bottom: what each one makes of one test image
(after ReLU). Some light up on the stroke, some on the empty background around it, some only on the
ends and corners of strokes.

In [ ]:
W1 = cnn[0].weight.detach()[:, 0]                       # (8, 3, 3)
img = Xte[3:4]
with torch.no_grad():
    maps = torch.relu(cnn[0](img))[0]                   # (8, 20, 20)
fig, axes = plt.subplots(2, 9, figsize=(13, 3.4), layout="constrained")
axes[0, 0].axis("off")
axes[1, 0].imshow(img[0, 0], cmap="Purples")
axes[1, 0].set_title(f"a {SHAPES[y_test[3]]}", fontsize=10)
axes[1, 0].axis("off")
for k in range(8):
    lim = W1[k].abs().max()
    axes[0, k + 1].imshow(W1[k], cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[0, k + 1].set_title(f"filter {k + 1}", fontsize=10)
    axes[1, k + 1].imshow(maps[k], cmap="Greens")
    for ax in axes[:, k + 1]:
        ax.axis("off")
plt.show()

## Your turn: how much does each part matter?
Build two smaller CNNs, train them with `train`, and print their weights and test accuracy:
(a) only the first convolution block, then `Flatten` and `Linear(8 * 10 * 10, 4)`;
(b) the full CNN with 2 and 4 filters instead of 8 and 16.

In [ ]:
def report(name, model):
    with torch.no_grad():
        acc = (model(Xte).argmax(dim=1) == yte).float().mean().item()
    print(f"{name:22s} {count(model):6,} weights   test accuracy {acc:.3f}")

torch.manual_seed(0)
one_block = None    # TODO: nn.Sequential(Conv2d(1, 8, 3, padding=1), ReLU, MaxPool2d(2), Flatten, Linear(8 * 10 * 10, 4))
torch.manual_seed(0)
thin = None         # TODO: like cnn, with 2 then 4 filters (and Linear(4 * 5 * 5, 4))
# TODO: train both, then report("one block", one_block) and report("2 and 4 filters", thin)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def report(name, model):
    with torch.no_grad():
        acc = (model(Xte).argmax(dim=1) == yte).float().mean().item()
    print(f"{name:22s} {count(model):6,} weights   test accuracy {acc:.3f}")

torch.manual_seed(0)
one_block = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(8 * 10 * 10, 4))
torch.manual_seed(0)
thin = nn.Sequential(nn.Conv2d(1, 2, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                     nn.Conv2d(2, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(4 * 5 * 5, 4))
report("full CNN", cnn)
report("one block", train(one_block))
report("2 and 4 filters", train(thin))
print("Two thin blocks (500 weights) beat one wide block (3,284): the second block combines the first one's")
print("edges into corners and curves, and its pooling lets the network ignore where exactly they are.")